# Training, inference, and energy measurement

Notebook 04 remains responsible for model selection and predictive-performance evaluation. This notebook measures the computational cost of the selected Logistic Regression, Random Forest, and XGBoost models on Windows or Linux. It records Python-process runtime, CPU use, and RSS memory even when no energy counter is available. Actual sensor readings, CodeCarbon estimates, and manual external-meter readings are stored separately and never presented as interchangeable.

Run Notebook 04 first. Fresh models are fitted for training-cost runs; already-trained grouped-CV fold models are used for genuinely held-out inference. No tuning or feature selection is done here. CodeBERT remains a separate experiment.

In [1]:
from pathlib import Path
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "Data" / "Processed").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Run Notebook 03 and Notebook 04 first.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from measurement_utils import (
    create_manual_meter_template,
    initialize_resource_experiment,
    process_manual_meter_entries,
    run_codecarbon_estimates,
    run_codebert_resource_measurements,
    run_inference_measurements,
    run_training_measurements,
    save_resource_summaries,
)

TRAINING_REPETITIONS = 10
INFERENCE_REPETITIONS = 10
INFERENCE_PASSES_PER_REPETITION = 10
PROCESS_SAMPLING_INTERVAL_SECONDS = 0.1
RUN_CODECARBON_ESTIMATES = False  # Estimates are separate from sensor readings.
RUN_CODEBERT_RESOURCE_MEASUREMENTS = False  # Set True only when you want to rerun the long CPU measurement.
CODECARBON_REPETITIONS = 1

state = initialize_resource_experiment(
    ROOT,
    training_repetitions=TRAINING_REPETITIONS,
    inference_repetitions=INFERENCE_REPETITIONS,
    inference_passes=INFERENCE_PASSES_PER_REPETITION,
    sampling_interval_seconds=PROCESS_SAMPLING_INTERVAL_SECONDS,
)
RUN_ROOT = state["run_root"]
ENERGY_BACKEND = state["energy_counter"].backend
print("Run ID:", state["run_id"])
print("Run output:", RUN_ROOT)
print("Actual energy source:", ENERGY_BACKEND or "unavailable; runtime/CPU/RSS still measured")
print("OS:", state["environment"]["operating_system"])
print("Processor:", state["environment"]["processor"])
print("Physical cores / logical threads:", state["environment"]["physical_cpu_cores"], "/", state["environment"]["logical_cpu_threads"])
print("Total RAM (GB):", state["environment"]["total_ram_gb"])
print("GPU devices:", state["environment"]["windows_gpu_names"] or state["environment"]["pytorch_cuda_devices"] or "none detected")
print("Python:", state["environment"]["python_version"].split()[0])
print("Libraries:", state["environment"]["library_versions"])
print("Process CPU utilization uses process CPU-time deltas normalized to total logical CPU capacity; it is not system CPU utilization.")
print("RSS peaks are periodically sampled and may miss short-lived peaks.")

Run ID: 20261108T210712+0100_33d8ae23
Run output: D:\Habiba\Uni\W26\Bachelor Thesis\energy-aware-code-recommendation\Results\resource_measurements\20261108T210712+0100_33d8ae23
Actual energy source: unavailable; runtime/CPU/RSS still measured
OS: Windows-10-10.0.19045-SP0
Processor: Intel64 Family 6 Model 141 Stepping 1, GenuineIntel
Physical cores / logical threads: 8 / 16
Total RAM (GB): 15.737
GPU devices: ['Intel(R) UHD Graphics', 'NVIDIA GeForce RTX 3050 Laptop GPU']
Python: 3.14.6
Libraries: {'psutil': '7.2.2', 'numpy': '2.5.4', 'pandas': '3.1.1', 'scikit-learn': '1.9.1', 'xgboost': '3.4.1', 'torch': '2.15.0', 'transformers': '5.22.1', 'codecarbon': '3.3.2', 'pyarrow': '26.0.1'}
Process CPU utilization uses process CPU-time deltas normalized to total logical CPU capacity; it is not system CPU utilization.
RSS peaks are periodically sampled and may miss short-lived peaks.


## Training cost

Each repetition creates a new estimator with Notebook 04's selected hyperparameters and fits it to the same full prepared training data and selected feature columns. The Logistic Regression `StandardScaler` is inside the pipeline. Training measurements and same-duration idle baselines are exported separately. This does not tune or select models.

In [2]:
training_measurements, training_idle_measurements = run_training_measurements(state)
training_measurements.groupby(["dataset", "model"], as_index=False).agg(
    repetitions=("repetition", "nunique"),
    mean_runtime_seconds=("elapsed_seconds", "mean"),
    std_runtime_seconds=("elapsed_seconds", "std"),
    mean_process_cpu_pct=("average_process_cpu_utilization_pct_of_total", "mean"),
    mean_peak_process_memory_mb=("peak_process_memory_mb", "mean"),
    mean_memory_increase_mb=("memory_increase_during_task_mb", "mean"),
    mean_actual_energy_joules=("actual_energy_joules", "mean"),
    std_actual_energy_joules=("actual_energy_joules", "std"),
)

validated/logistic_regression training 1/10: 0.079s


validated/logistic_regression training 2/10: 0.041s
validated/logistic_regression training 3/10: 0.044s


validated/logistic_regression training 4/10: 0.042s


validated/logistic_regression training 5/10: 0.045s
validated/logistic_regression training 6/10: 0.043s


validated/logistic_regression training 7/10: 0.042s


validated/logistic_regression training 8/10: 0.044s
validated/logistic_regression training 9/10: 0.042s


... (39 outputs left out)

significant/xgboost training 8/10: 0.112s


significant/xgboost training 9/10: 0.113s


significant/xgboost training 10/10: 0.117s


,dataset,model,repetitions,mean_runtime_seconds,std_runtime_seconds,mean_process_cpu_pct,mean_peak_process_memory_mb,mean_memory_increase_mb,mean_actual_energy_joules,std_actual_energy_joules
0,significant,logistic_regression,10,0.025233,0.000640,4.721576,650.942188,0.000000,NaN,NaN
1,significant,random_forest,10,0.372935,0.010277,17.404110,656.405859,1.217969,NaN,NaN
2,significant,xgboost,10,0.113823,0.003370,24.345405,657.581641,1.511328,NaN,NaN
3,validated,logistic_regression,10,0.046385,0.011430,4.303482,548.026953,0.083984,NaN,NaN
4,validated,random_forest,10,0.538237,0.018064,29.912502,589.541406,35.436719,NaN,NaN
5,validated,xgboost,10,0.232795,0.402598,21.565920,651.198828,9.702344,NaN,NaN


## Held-out inference cost

Inference uses the saved Notebook 04 fold models and only each fold's out-of-fold problem groups. The model is loaded before timing; the prediction call is timed, and inference never retrains the model. The full held-out fold batches are repeated to make timings measurable. The saved per-repetition CSV reports unique samples, total predictions, time/sample, and throughput.

In [3]:
inference_measurements, inference_fold_measurements, inference_idle_measurements = run_inference_measurements(state)
inference_measurements.groupby(["dataset", "model"], as_index=False).agg(
    repetitions=("repetition", "nunique"),
    mean_runtime_seconds=("elapsed_seconds", "mean"),
    std_runtime_seconds=("elapsed_seconds", "std"),
    mean_time_per_prediction_seconds=("average_inference_time_per_sample_seconds", "mean"),
    mean_predictions_per_second=("predictions_per_second", "mean"),
    mean_process_cpu_pct=("average_process_cpu_utilization_pct_of_total", "mean"),
    mean_peak_process_memory_mb=("peak_process_memory_mb", "mean"),
    mean_actual_energy_joules=("actual_energy_joules", "mean"),
    std_actual_energy_joules=("actual_energy_joules", "std"),
)

validated/logistic_regression inference 1/10: 0.059s for 214,100 predictions


validated/logistic_regression inference 2/10: 0.054s for 214,100 predictions


validated/logistic_regression inference 3/10: 0.055s for 214,100 predictions


validated/logistic_regression inference 4/10: 0.054s for 214,100 predictions


validated/logistic_regression inference 5/10: 0.056s for 214,100 predictions


validated/logistic_regression inference 6/10: 0.054s for 214,100 predictions


... (46 outputs left out)

significant/xgboost inference 8/10: 0.146s for 30,000 predictions


significant/xgboost inference 9/10: 0.150s for 30,000 predictions


significant/xgboost inference 10/10: 0.148s for 30,000 predictions


,dataset,model,repetitions,mean_runtime_seconds,std_runtime_seconds,mean_time_per_prediction_seconds,mean_predictions_per_second,mean_process_cpu_pct,mean_peak_process_memory_mb,mean_actual_energy_joules,std_actual_energy_joules
0,significant,logistic_regression,10,0.053384,0.000795,1.779454e-06,5.620817e+05,6.654091,661.199219,NaN,NaN
1,significant,random_forest,10,1.989610,0.005901,6.632035e-05,1.507845e+04,7.038011,671.076953,NaN,NaN
2,significant,xgboost,10,0.147318,0.001523,4.910589e-06,2.036612e+05,30.556036,658.097266,NaN,NaN
3,validated,logistic_regression,10,0.056113,0.002563,2.620865e-07,3.822489e+06,6.037727,656.394531,NaN,NaN
4,validated,random_forest,10,2.014031,0.021210,9.406966e-06,1.063148e+05,15.050931,704.099219,NaN,NaN
5,validated,xgboost,10,0.181386,0.003201,8.472038e-07,1.180676e+06,25.995790,660.964062,NaN,NaN


## Optional HTW Berlin external power-meter entry

No Python API is assumed. The first run creates a blank CSV template in the run folder. Fill it with the meter's initial/final readings, unit, duration, experiment, model, task, and repetition, then execute this cell again. Supported units are J, Wh, kWh, and mWh. Output energy is explicitly whole-system wall energy, not CPU/package energy.

In [4]:
meter_template = create_manual_meter_template(RUN_ROOT)
meter_results_path = RUN_ROOT / "external_power_meter_manual_results.csv"
manual_meter_measurements = process_manual_meter_entries(meter_template, meter_results_path)
if manual_meter_measurements.empty:
    print("Manual meter template:", meter_template)
    print("No meter entries yet; fill the template and rerun this cell to convert readings to Joules.")
else:
    display(manual_meter_measurements)

Manual meter template: D:\Habiba\Uni\W26\Bachelor Thesis\energy-aware-code-recommendation\Results\resource_measurements\20261108T210712+0100_33d8ae23\external_power_meter_manual_entry.csv
No meter entries yet; fill the template and rerun this cell to convert readings to Joules.


## Optional CodeCarbon estimates

CodeCarbon is opt-in and reports estimates, not hardware-sensor measurements. When enabled, its training and inference estimates are written to separate CSVs under each dataset/task. They are never merged with RAPL, Scaphandre, or manual-meter values.

In [5]:
if RUN_CODECARBON_ESTIMATES:
    codecarbon_measurements = run_codecarbon_estimates(state, repetitions=CODECARBON_REPETITIONS)
    display(codecarbon_measurements.groupby(["dataset", "model", "task"], as_index=False).agg(
        repetitions=("repetition", "count"),
        mean_estimated_energy_joules=("estimated_energy_joules", "mean"),
        std_estimated_energy_joules=("estimated_energy_joules", "std"),
    ))
else:
    codecarbon_measurements = None
    print("CodeCarbon disabled. Set RUN_CODECARBON_ESTIMATES = True to opt in; estimates stay separate.")

CodeCarbon disabled. Set RUN_CODECARBON_ESTIMATES = True to opt in; estimates stay separate.


## Summaries and output files

Every execution receives a new timestamped run directory, so previous experiments are not overwritten. Validated and significant results are kept in separate folders, each with `training/` and `inference/` details, idle baselines, summaries, and status metadata. `environment.json` and `run_metadata.json` capture the platform, library versions, selected features, settings, and repetitions. Blank actual-energy fields mean no hardware energy sensor was available; they are not zero-energy readings.

In [6]:
state["run_metadata"]["codecarbon_estimates_enabled"] = RUN_CODECARBON_ESTIMATES
state["run_metadata"]["codecarbon_repetitions"] = CODECARBON_REPETITIONS if RUN_CODECARBON_ESTIMATES else 0
training_summary, inference_summary = save_resource_summaries(state)
print("Saved new run under:", RUN_ROOT)
print("Training summary:")
display(training_summary)
print("Inference summary:")
display(inference_summary)
print("Actual energy:", state["energy_counter"].backend or "unavailable")

Saved new run under: D:\Habiba\Uni\W26\Bachelor Thesis\energy-aware-code-recommendation\Results\resource_measurements\20261108T210712+0100_33d8ae23
Training summary:


,dataset,model,feature_config,repetitions,mean_runtime_seconds,std_runtime_seconds,mean_process_cpu_utilization_pct_of_total,mean_peak_process_memory_mb,std_peak_process_memory_mb,mean_memory_increase_mb,mean_actual_energy_joules,std_actual_energy_joules,mean_net_actual_energy_joules,std_net_actual_energy_joules
0,significant,logistic_regression,combined,10,0.025233,0.000640,4.721576,650.942188,0.014823,0.000000,NaN,NaN,NaN,NaN
1,significant,random_forest,combined,10,0.372935,0.010277,17.404110,656.405859,0.419374,1.217969,NaN,NaN,NaN,NaN
2,significant,xgboost,combined,10,0.113823,0.003370,24.345405,657.581641,1.279839,1.511328,NaN,NaN,NaN,NaN
3,validated,logistic_regression,config_B,10,0.046385,0.011430,4.303482,548.026953,0.001235,0.083984,NaN,NaN,NaN,NaN
4,validated,random_forest,config_B,10,0.538237,0.018064,29.912502,589.541406,1.439467,35.436719,NaN,NaN,NaN,NaN
5,validated,xgboost,config_B,10,0.232795,0.402598,21.565920,651.198828,0.619571,9.702344,NaN,NaN,NaN,NaN


Inference summary:


,dataset,model,feature_config,repetitions,mean_runtime_seconds,std_runtime_seconds,mean_process_cpu_utilization_pct_of_total,mean_peak_process_memory_mb,std_peak_process_memory_mb,mean_memory_increase_mb,mean_actual_energy_joules,std_actual_energy_joules,mean_net_actual_energy_joules,std_net_actual_energy_joules,mean_seconds_per_prediction,mean_predictions_per_second
0,significant,logistic_regression,combined,10,0.053384,0.000795,6.654091,661.199219,0.000000,0.000000,NaN,NaN,NaN,NaN,1.779454e-06,5.620817e+05
1,significant,random_forest,combined,10,1.989610,0.005901,7.038011,671.076953,0.291167,0.685937,NaN,NaN,NaN,NaN,6.632035e-05,1.507845e+04
2,significant,xgboost,combined,10,0.147318,0.001523,30.556036,658.097266,0.262779,0.000000,NaN,NaN,NaN,NaN,4.910589e-06,2.036612e+05
3,validated,logistic_regression,config_B,10,0.056113,0.002563,6.037727,656.394531,0.000000,0.000000,NaN,NaN,NaN,NaN,2.620865e-07,3.822489e+06
4,validated,random_forest,config_B,10,2.014031,0.021210,15.050931,704.099219,0.314335,1.974219,NaN,NaN,NaN,NaN,9.406966e-06,1.063148e+05
5,validated,xgboost,config_B,10,0.181386,0.003201,25.995790,660.964062,0.593467,0.002734,NaN,NaN,NaN,NaN,8.472038e-07,1.180676e+06


Actual energy: unavailable


## CodeBERT: separate from tabular measurements

The existing CodeBERT experiments are preserved in Notebook 04 and `experiment_utils.py`. The encoder is frozen and used for embedding extraction with a Logistic Regression head; it is not fine-tuned. Its training includes embedding extraction, while inference includes code encoding and prediction. Resource measurements force CodeBERT onto the CPU even when CUDA is available; GPU power and memory are not measured. Actual energy is reported only when a supported energy counter is available. These results remain separate from the tabular summaries.

In [2]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents) if (path / "Data" / "Processed").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Run Notebook 03 and Notebook 04 first.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from measurement_utils import initialize_resource_experiment, run_codebert_resource_measurements

RUN_CODEBERT_RESOURCE_MEASUREMENTS = False  # Set True to rerun; CPU training can take over an hour.

if RUN_CODEBERT_RESOURCE_MEASUREMENTS:
    if "state" not in globals():
        state = initialize_resource_experiment(
            ROOT,
            training_repetitions=1,
            inference_repetitions=10,
            inference_passes=1,
            sampling_interval_seconds=0.1,
        )
    codebert_resource_measurements = run_codebert_resource_measurements(
        state,
        training_repetitions=1,
        inference_repetitions=10,
    )
else:
    dataset_names = ("validated", "significant")
    result_dirs = sorted(
        (ROOT / "Results" / "resource_measurements").glob("*/codebert"),
        reverse=True,
    )
    completed_dir = next(
        (
            path for path in result_dirs
            if all(
                (path / name / "codebert_resource_measurements.csv").is_file()
                for name in dataset_names
            )
        ),
        None,
    )
    if completed_dir is None:
        raise FileNotFoundError(
            "No complete saved CodeBERT measurements found. Set "
            "RUN_CODEBERT_RESOURCE_MEASUREMENTS=True to run them."
        )
    codebert_resource_measurements = pd.concat(
        [
            pd.read_csv(
                completed_dir / name / "codebert_resource_measurements.csv"
            )
            for name in dataset_names
        ],
        ignore_index=True,
    )
    print(f"Loaded completed CodeBERT measurements from {completed_dir.parent}.")

display(codebert_resource_measurements.groupby(["dataset", "model", "task"], as_index=False).agg(
    repetitions=("repetition", "nunique"),
    measurement_device=("measurement_device", "first"),
    mean_runtime_seconds=("elapsed_seconds", "mean"),
    mean_process_cpu_pct=("average_process_cpu_utilization_pct_of_total", "mean"),
    mean_peak_process_memory_mb=("peak_process_memory_mb", "mean"),
    mean_actual_energy_joules=("actual_energy_joules", "mean"),
))


Loaded completed CodeBERT measurements from d:\Habiba\Uni\W26\Bachelor Thesis\energy-aware-code-recommendation\Results\resource_measurements\20261109T215938+0100_c184ff76.


,dataset,model,task,repetitions,measurement_device,mean_runtime_seconds,mean_process_cpu_pct,mean_peak_process_memory_mb,mean_actual_energy_joules
0,significant,codebert_frozen_embeddings_logistic_regression,inference,10,cpu,60.551736,49.939790,991.327734,NaN
1,significant,codebert_frozen_embeddings_logistic_regression,training,1,cpu,719.711757,49.915737,1067.093750,NaN
2,validated,codebert_frozen_embeddings_logistic_regression,inference,10,cpu,71.783552,49.986926,1066.035938,NaN
3,validated,codebert_frozen_embeddings_logistic_regression,training,1,cpu,5192.479053,49.986719,1887.316406,NaN
